In [1]:
import pandas as pd
import numpy as np
import re
import csv
import nltk
from nltk.corpus import stopwords
import pymorphy3
from sklearn.feature_extraction.text import TfidfVectorizer

предобработка - сделать лучше (убрать английские, свернуть междометия, стоп-слова уалять не все) +
словарь tfidf - проверить, что с ним не так
на 7 признаках обучить классификатор


# 1.	Объединить позитивный и негативный корпуса RuTweetCorp, например, используя pd.concat(…).


In [2]:
# Загружаем файлы с позитивными и негативными твитами
df_negative_twits = pd.read_csv('negative.csv', 
                     sep=';',              
                     quoting=csv.QUOTE_ALL, # поля в кавычках!
                     header=None,           
                     encoding='utf-8')      

df_positive_twits = pd.read_csv('positive.csv', 
                     sep=';', 
                     quoting=csv.QUOTE_ALL, 
                     header=None, 
                     encoding='utf-8')

In [3]:
df_negative_twits.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0


In [4]:
df_positive_twits.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11
0,408906692374446080,1386325927,pleease_shut_up,"@first_timee хоть я и школота, но поверь, у на...",1,0,0,0,7569,62,61,0
1,408906692693221377,1386325927,alinakirpicheva,"Да, все-таки он немного похож на него. Но мой ...",1,0,0,0,11825,59,31,2
2,408906695083954177,1386325927,EvgeshaRe,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,0,1,0,1273,26,27,0


In [5]:
#имена столбцов
columnes = ['tw_id', 'date', 'user_name', 'text', 'tone', 'reply', 
                'favorite', 'retwit', 'followers', 'friends', 
                'statuses', 'tag_user']


In [6]:
df_negative_twits.columns = columnes

In [7]:
df_negative_twits.head(4)


,tw_id,date,user_name,text,tone,reply,favorite,retwit,followers,friends,statuses,tag_user
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0
3,408906914437685248,1386325980,Poliwake,"Желаю хорошего полёта и удачной посадки,я буду...",-1,0,0,0,10628,207,200,0


In [8]:
df_positive_twits.columns = columnes

In [9]:
df_positive_twits.head(4)

,tw_id,date,user_name,text,tone,reply,favorite,retwit,followers,friends,statuses,tag_user
0,408906692374446080,1386325927,pleease_shut_up,"@first_timee хоть я и школота, но поверь, у на...",1,0,0,0,7569,62,61,0
1,408906692693221377,1386325927,alinakirpicheva,"Да, все-таки он немного похож на него. Но мой ...",1,0,0,0,11825,59,31,2
2,408906695083954177,1386325927,EvgeshaRe,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,0,1,0,1273,26,27,0
3,408906695356973056,1386325927,ikonnikova_21,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,0,1,0,1549,19,17,0


In [10]:
print('Негативных твитов: ' + str(len(df_negative_twits)))

Негативных твитов: 111923


In [11]:
print('Позитивных твитов: ' + str(len(df_positive_twits)))

Позитивных твитов: 114911


In [12]:
df_all_twits = pd.concat([df_negative_twits, df_positive_twits], ignore_index=True)

In [13]:
print('Твитов всего: ' + str(len(df_all_twits)))

Твитов всего: 226834


In [14]:
df_all_twits.head(5)

,tw_id,date,user_name,text,tone,reply,favorite,retwit,followers,friends,statuses,tag_user
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0
3,408906914437685248,1386325980,Poliwake,"Желаю хорошего полёта и удачной посадки,я буду...",-1,0,0,0,10628,207,200,0
4,408906914723295232,1386325980,capyvixowe,"Обновил за каким-то лешим surf, теперь не рабо...",-1,0,0,0,35,17,34,0


# 2.	Удалить стоп-слова, используя предустановленный список стоп-слов в библиотеке NLTK
2.1 Привести текст к нижнему регистру

2.2 Токенизировать по словам

2.3 Удалить стоп-слова

2.4 Удалить знаки пунктуации и ссылки

In [18]:
nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))
print(list(russian_stopwords))

['сам', 'было', 'вы', 'разве', 'всех', 'тебя', 'вам', 'под', 'если', 'чуть', 'всегда', 'моя', 'о', 'эту', 'всего', 'то', 'быть', 'тогда', 'этой', 'там', 'через', 'все', 'был', 'такой', 'нибудь', 'него', 'но', 'ему', 'при', 'нельзя', 'себе', 'всю', 'между', 'бы', 'ним', 'почти', 'три', 'какой', 'вас', 'мне', 'свою', 'ничего', 'ней', 'чем', 'вдруг', 'один', 'во', 'с', 'в', 'меня', 'может', 'потом', 'что', 'их', 'без', 'ведь', 'много', 'ей', 'надо', 'тут', 'над', 'два', 'ты', 'для', 'какая', 'потому', 'и', 'только', 'или', 'по', 'перед', 'она', 'нет', 'вот', 'ее', 'мой', 'нее', 'к', 'чего', 'них', 'где', 'есть', 'тем', 'из', 'его', 'им', 'да', 'же', 'том', 'хоть', 'никогда', 'тоже', 'я', 'ну', 'мы', 'другой', 'нас', 'ли', 'опять', 'до', 'чтоб', 'конечно', 'когда', 'наконец', 'была', 'можно', 'на', 'того', 'эти', 'куда', 'этот', 'больше', 'хорошо', 'иногда', 'у', 'уже', 'этого', 'здесь', 'были', 'ж', 'как', 'так', 'совсем', 'лучше', 'раз', 'тот', 'со', 'себя', 'уж', 'даже', 'а', 'ни', 'кто

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [19]:
print(f"Всего стоп-слов: {len(russian_stopwords)}")

Всего стоп-слов: 151


In [20]:
my_stopwords = russian_stopwords.copy()
words_to_remove = {'вдруг', 'конечно', 'чего', 'хоть', 'всего', 'какой', 'всегда', 'зачем', 'наконец', 'лучше', 'моя', 'ничего', 'хорошо', 'меня', 'даже', 'этот', 'эта', 'два', 'ней', 'и', 'того', 'какая', 'будто', 'только', 'никогда', 'теперь', 'потом', 'нельзя', 'можно', 'этой', 'свою', 'более', 'где', 'тогда', 'раз'}

# Удаляем слова по одному
for word in words_to_remove:
    if word in my_stopwords:  # проверяем, есть ли слово
        my_stopwords.remove(word)
        print(f"Удалили: {word}")
    else:
        print(f"Слово '{word}' не найдено в стоп-словах")

Удалили: наконец
Удалили: ничего
Удалили: ней
Удалили: можно
Удалили: вдруг
Удалили: того
Удалили: этот
Удалили: всегда
Удалили: хорошо
Удалили: моя
Удалили: чего
Удалили: меня
Удалили: только
Удалили: потом
Удалили: где
Удалили: всего
Слово 'эта' не найдено в стоп-словах
Удалили: тогда
Удалили: этой
Удалили: лучше
Удалили: раз
Удалили: даже
Удалили: хоть
Удалили: нельзя
Удалили: теперь
Удалили: никогда
Удалили: будто
Удалили: более
Удалили: два
Удалили: какая
Удалили: какой
Удалили: и
Удалили: свою
Удалили: конечно
Удалили: зачем


In [21]:
print(f"Всего стоп-слов теперь: {len(my_stopwords)}")

Всего стоп-слов теперь: 117


In [22]:
print("Примеры стоп-слов:", list(my_stopwords))

Примеры стоп-слов: ['сам', 'было', 'вы', 'разве', 'всех', 'тебя', 'вам', 'под', 'если', 'чуть', 'о', 'эту', 'то', 'быть', 'там', 'через', 'все', 'был', 'такой', 'нибудь', 'него', 'но', 'ему', 'при', 'себе', 'всю', 'между', 'бы', 'ним', 'почти', 'три', 'вас', 'мне', 'чем', 'один', 'во', 'с', 'в', 'может', 'что', 'их', 'без', 'ведь', 'много', 'ей', 'надо', 'тут', 'над', 'ты', 'для', 'потому', 'или', 'по', 'перед', 'она', 'нет', 'вот', 'ее', 'мой', 'нее', 'к', 'них', 'есть', 'тем', 'из', 'его', 'им', 'да', 'же', 'том', 'тоже', 'я', 'ну', 'мы', 'другой', 'нас', 'ли', 'опять', 'до', 'чтоб', 'когда', 'была', 'на', 'эти', 'куда', 'больше', 'иногда', 'у', 'уже', 'этого', 'здесь', 'были', 'ж', 'как', 'так', 'совсем', 'тот', 'со', 'себя', 'уж', 'а', 'ни', 'кто', 'за', 'после', 'он', 'сейчас', 'они', 'от', 'впрочем', 'будет', 'этом', 'чтобы', 'не', 'про', 'об', 'еще']


In [32]:
plus_stopwords = {'http', 'https', 'из', 'за', 'без', 'ага', 'аж'}
my_stopwords.update(plus_stopwords)
print(f"Всего стоп-слов: {len(my_stopwords)}")
print("Примеры стоп-слов:", list(my_stopwords))

Всего стоп-слов: 121
Примеры стоп-слов: ['сам', 'было', 'вы', 'разве', 'всех', 'тебя', 'вам', 'под', 'если', 'чуть', 'о', 'эту', 'то', 'быть', 'там', 'через', 'все', 'был', 'такой', 'нибудь', 'него', 'но', 'ему', 'при', 'себе', 'всю', 'между', 'бы', 'ним', 'почти', 'три', 'вас', 'мне', 'чем', 'один', 'во', 'с', 'в', 'может', 'что', 'аж', 'их', 'без', 'ведь', 'много', 'https', 'ей', 'надо', 'тут', 'над', 'ты', 'для', 'потому', 'или', 'по', 'перед', 'она', 'нет', 'вот', 'ее', 'мой', 'нее', 'к', 'них', 'есть', 'тем', 'из', 'его', 'им', 'да', 'же', 'том', 'тоже', 'я', 'ну', 'мы', 'другой', 'нас', 'ли', 'опять', 'до', 'чтоб', 'http', 'когда', 'была', 'на', 'эти', 'куда', 'больше', 'иногда', 'у', 'уже', 'этого', 'здесь', 'были', 'ж', 'как', 'так', 'совсем', 'тот', 'со', 'себя', 'уж', 'а', 'ни', 'кто', 'за', 'ага', 'после', 'он', 'сейчас', 'они', 'от', 'впрочем', 'будет', 'этом', 'чтобы', 'не', 'про', 'об', 'еще']


# 3. Очистка текста

In [71]:
# удаляем ВСЕ, что не является чистым русским текстом
def cleaner_text_strict(text: str, stopwords_set: Set[str]) -> List[str]:
    """
    Строгая версия - только чистые русские слова длиной 3+ буквы
    """
    if not isinstance(text, str) or not text.strip():
        return []
    
    # Очистка
    text = re.sub(r'https?://\S+|www\.\S+|t\.co/\S+|@\w+|[^\w\s]|\d+', ' ', text.lower())
    words = text.split()
    
    filtered_words = []
    
    for word in words:
        # Если есть английские буквы - сразу пропускаем
        if re.search(r'[a-z]', word):
            continue
        
        # Только русские буквы
        if not re.match(r'^[а-яё]+$', word):
            continue
        
        # Только слова длиной 3+ буквы
        if len(word) < 3:
            continue
        
        # Не стоп-слова
        if word in stopwords_set:
            continue
        
        # Не повторы букв
        if len(set(word)) == 1:
            continue
        
        filtered_words.append(word)
    
    return filtered_words

In [73]:
df_all_twits.head(4)

,tw_id,date,user_name,text,tone,reply,favorite,retwit,followers,friends,statuses,tag_user
0,408906762813579264,1386325944,dugarchikbellko,на работе был полный пиддес :| и так каждое за...,-1,0,0,0,8064,111,94,2
1,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
2,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0
3,408906914437685248,1386325980,Poliwake,"Желаю хорошего полёта и удачной посадки,я буду...",-1,0,0,0,10628,207,200,0


In [74]:
def apply_cleaner_text(text):
    """Применяет cleaner_text с plus_stopwords к одному тексту"""
    return cleaner_text(text, plus_stopwords)

def join_tokens(tokens):
    """Объединяет список токенов в строку"""
    if tokens:
        return ' '.join(tokens)
    return ''

def count_tokens(text):
    """Считает количество токенов в тексте"""
    if pd.isna(text) or not isinstance(text, str):
        return 0
    return len(str(text).split())

def count_list_length(lst):
    """Считает длину списка"""
    if isinstance(lst, list):
        return len(lst)
    return 0

# Создаем новый датафрейм с нужными колонками
df_all_twits_text = pd.DataFrame()
df_all_twits_text['text'] = df_all_twits['text']
df_all_twits_text['tokens'] = df_all_twits['text'].apply(apply_cleaner_text)
df_all_twits_text['cleaned_text'] = df_all_twits_text['tokens'].apply(join_tokens)
df_all_twits_text['tone']=df_all_twits['tone']

# Выводим информацию о новом датафрейме
print("\n" + "="*50)
print("ИНФОРМАЦИЯ О НОВОМ ДАТАФРЕЙМЕ")
print("="*50)
print(f"Размер датафрейма: {df_all_twits_text.shape}")
print(f"Колонки: {df_all_twits_text.columns.tolist()}")
print(f"Количество строк: {len(df_all_twits_text)}")

# Проверяем результат на первых строках
print("\n" + "="*50)
print("ПРИМЕРЫ ОБРАБОТКИ (первые 5 строк)")
print("="*50)

for i in range(min(5, len(df_all_twits_text))):
    print(f"\n--- Строка {i} ---")
    
    # Исходный текст
    original_text = df_all_twits_text['text'].iloc[i]
    if isinstance(original_text, str):
        print(f"Исходный текст: {original_text[:100]}...")
    else:
        print(f"Исходный текст: {original_text}")
    
    # Токены
    tokens = df_all_twits_text['tokens'].iloc[i]
    if isinstance(tokens, list):
        print(f"Токены: {tokens[:10]}...")  # первые 10 токенов
    else:
        print(f"Токены: {tokens}")
    
    # Очищенный текст
    cleaned = df_all_twits_text['cleaned_text'].iloc[i]
    if isinstance(cleaned, str):
        print(f"Очищенный текст: {cleaned[:100]}...")
    else:
        print(f"Очищенный текст: {cleaned}")

# Статистика по очистке
print("\n" + "="*50)
print("СТАТИСТИКА ОЧИСТКИ")
print("="*50)

# Количество токенов до и после
df_all_twits_text['original_tokens'] = df_all_twits['text'].apply(count_tokens)
df_all_twits_text['cleaned_tokens_count'] = df_all_twits_text['tokens'].apply(count_list_length)

print(f"Всего исходных токенов: {df_all_twits_text['original_tokens'].sum()}")
print(f"Всего токенов после очистки: {df_all_twits_text['cleaned_tokens_count'].sum()}")

# Избегаем деления на ноль
if df_all_twits_text['original_tokens'].sum() > 0:
    percent_saved = df_all_twits_text['cleaned_tokens_count'].sum() / df_all_twits_text['original_tokens'].sum() * 100
    print(f"Процент сохраненных токенов: {percent_saved:.1f}%")
else:
    print("Процент сохраненных токенов: нет исходных токенов")

print(f"Среднее количество токенов ДО очистки: {df_all_twits_text['original_tokens'].mean():.1f}")
print(f"Среднее количество токенов ПОСЛЕ очистки: {df_all_twits_text['cleaned_tokens_count'].mean():.1f}")

# Проверка на пустые результаты
empty_count = (df_all_twits_text['cleaned_tokens_count'] == 0).sum()
print(f"\nТекстов, полностью очищенных (пустой результат): {empty_count}")
print(f"Процент пустых результатов: {empty_count/len(df_all_twits_text)*100:.1f}%")

# Сохраняем результат
df_all_twits_text.to_csv('df_all_twits_text.csv', index=False, encoding='utf-8')


ИНФОРМАЦИЯ О НОВОМ ДАТАФРЕЙМЕ
Размер датафрейма: (226834, 4)
Колонки: ['text', 'tokens', 'cleaned_text', 'tone']
Количество строк: 226834

ПРИМЕРЫ ОБРАБОТКИ (первые 5 строк)

--- Строка 0 ---
Исходный текст: на работе был полный пиддес :| и так каждое закрытие месяца, я же свихнусь так D:...
Токены: ['на', 'работе', 'был', 'полный', 'пиддес', 'так', 'каждое', 'закрытие', 'месяца', 'же']...
Очищенный текст: на работе был полный пиддес так каждое закрытие месяца же свихнусь так...

--- Строка 1 ---
Исходный текст: Коллеги сидят рубятся в Urban terror, а я из-за долбанной винды не могу :(...
Токены: ['коллеги', 'сидят', 'рубятся', 'долбанной', 'винды', 'не', 'могу']...
Очищенный текст: коллеги сидят рубятся долбанной винды не могу...

--- Строка 2 ---
Исходный текст: @elina_4post как говорят обещаного три года ждут...((...
Токены: ['как', 'говорят', 'обещаного', 'три', 'года', 'ждут']...
Очищенный текст: как говорят обещаного три года ждут...

--- Строка 3 ---
Исходный текст: Желаю хорош

In [75]:
df_all_twits_text.head(4)

,text,tokens,cleaned_text,tone,original_tokens,cleaned_tokens_count
0,на работе был полный пиддес :| и так каждое за...,"[на, работе, был, полный, пиддес, так, каждое,...",на работе был полный пиддес так каждое закрыти...,-1,16,12
1,"Коллеги сидят рубятся в Urban terror, а я из-з...","[коллеги, сидят, рубятся, долбанной, винды, не...",коллеги сидят рубятся долбанной винды не могу,-1,14,7
2,@elina_4post как говорят обещаного три года жд...,"[как, говорят, обещаного, три, года, ждут]",как говорят обещаного три года ждут,-1,7,6
3,"Желаю хорошего полёта и удачной посадки,я буду...","[желаю, хорошего, полёта, удачной, посадки, бу...",желаю хорошего полёта удачной посадки буду оче...,-1,11,9


# 4 Словарь linis_crowd

In [ ]:
linis_crowd = pd.read_csv('words_all_full_rating_utf8.csv', sep = ';', quoting=csv.QUOTE_ALL, encoding = 'utf-8')

In [ ]:
linis_crowd

In [76]:
tonal_dict = {}
for idx, row in linis_crowd.iterrows():
    word = row['Words']
    mean_str = str(row['mean']).replace(',', '.')
    try:
        tonal_dict[word] = float(mean_str)
    except ValueError:
        tonal_dict[word] = 0.0
print(f"Создан словарь эмоциональной лексики из {len(tonal_dict)} слов")

Создан словарь эмоциональной лексики из 7545 слов


# 3.	Извлечь лингвистические признаки

Используя тональный словарь LinisCrowd 2015 (http://linis-crowd.org/), действовать по следующему алгоритму:

 3.1) токенизация - сделана выше;
 
 3.2) лемматизация;
 
 3.3) определение токенов (после лемматизации) в сообщениях, которые есть в тональном словаре и формирование списка средних значений тональности таких слов.


In [77]:
morph = pymorphy3.MorphAnalyzer()

In [78]:
# 3.2 Лемматизация
def lemmatization_tokens(tokens):
    lemmas = []
    for token in tokens:
        parsed = morph.parse(token)[0] 
        lemmas.append(parsed.normal_form)
    return lemmas

In [79]:
# 3.3 Определение токенов в тональном словаре и формирование списка тональных слов
def extract_tonal_features_with_lemmas(tokens, tonal_dict):
    # лемматизируем сами токены
    lemmas = lemmatization_tokens(tokens)
    
    # Ищем леммы в тональном словаре
    tonal_values = []
    words_found = []
    
    for lemma in lemmas:
        if lemma in tonal_dict:
            tonal_values.append(tonal_dict[lemma])
            words_found.append(lemma)
    
    # Если слова не найдены, возвращаем [0]
    if not tonal_values:
        return [0], [], lemmas
    
    # Иначе возвращаем список тональностей как строки
    return [str(v) for v in tonal_values], words_found, lemmas

# Функция для обработки одного твита 
def process_twit(tokens, dict_for_search):
    return extract_tonal_features_with_lemmas(tokens, dict_for_search)

# Применяем ко всем твитам

results = df_all_twits_text['tokens'].apply(lambda x: process_twit(x, tonal_dict))

# Распаковываем результаты в колонки датафрейма
df_all_twits_text['tonal_list'] = [r[0] for r in results]      # тональности
df_all_twits_text['found_lemmas'] = [r[1] for r in results]    # нашел эти леммы
df_all_twits_text['lemmas'] = [r[2] for r in results]          # все леммы твита

print(f"Обработано твитов: {len(df_all_twits_text)}")
df_all_twits_text[['tokens', 'lemmas', 'found_lemmas', 'tonal_list']].head(3)

Обработано твитов: 226834


,tokens,lemmas,found_lemmas,tonal_list
0,"[на, работе, был, полный, пиддес, так, каждое,...","[на, работа, быть, полный, пиддес, так, каждый...","[работа, полный, закрытие]","[0.166666666666667, 0.222222222222222, 0.0]"
1,"[коллеги, сидят, рубятся, долбанной, винды, не...","[коллега, сидеть, рубиться, долбать, винд, не,...",[сидеть],[-0.2]
2,"[как, говорят, обещаного, три, года, ждут]","[как, говорить, обещаной, три, год, ждать]",[ждать],[0.0]



  3.4 Из списка тональностей взять:
    - среднее значение
    - максимальное значение
    - минимальное значение
    - суммарное значение
    - количество положительных значений (>0)
    - количество отрицательных значений (<0)
    
 Если список пустой или [0], возвращаем [0, 0, 0, 0, 0, 0]
  

In [80]:
# Берем признаки из списка тональностей

def calculate_features_from_tonal_list(tonal_list):

    if tonal_list == [0] or not tonal_list:
        return [0.0, 0.0, 0.0, 0.0, 0, 0]
    
    values = [float(v) for v in tonal_list]
    
    #Эти признаки
    mean_val = sum(values) / len(values)           
    max_val = max(values)                          
    min_val = min(values)                           
    sum_val = sum(values)  
    
    # не путать тональности
    pos_count = sum(1 for v in values if v > 0)
    neg_count = sum(1 for v in values if v < 0)
    
    return [mean_val, max_val, min_val, sum_val, pos_count, neg_count]

In [81]:
# Применяем функцию ко всем твитам
features_results = df_all_twits_text['tonal_list'].apply(calculate_features_from_tonal_list)

# Добавить признаки в датафрейм
df_all_twits_text[['tone_mean', 'tone_max', 'tone_min', 'tone_sum', 
              'tone_pos_count', 'tone_neg_count']] = pd.DataFrame(
    features_results.tolist(), index=df_all_twits_text.index
)

In [82]:
df_all_twits_text.head(7)

,text,tokens,cleaned_text,tone,original_tokens,cleaned_tokens_count,tonal_list,found_lemmas,lemmas,tone_mean,tone_max,tone_min,tone_sum,tone_pos_count,tone_neg_count
0,на работе был полный пиддес :| и так каждое за...,"[на, работе, был, полный, пиддес, так, каждое,...",на работе был полный пиддес так каждое закрыти...,-1,16,12,"[0.166666666666667, 0.222222222222222, 0.0]","[работа, полный, закрытие]","[на, работа, быть, полный, пиддес, так, каждый...",0.129630,0.222222,0.00,0.388889,2,0
1,"Коллеги сидят рубятся в Urban terror, а я из-з...","[коллеги, сидят, рубятся, долбанной, винды, не...",коллеги сидят рубятся долбанной винды не могу,-1,14,7,[-0.2],[сидеть],"[коллега, сидеть, рубиться, долбать, винд, не,...",-0.200000,-0.200000,-0.20,-0.200000,0,1
2,@elina_4post как говорят обещаного три года жд...,"[как, говорят, обещаного, три, года, ждут]",как говорят обещаного три года ждут,-1,7,6,[0.0],[ждать],"[как, говорить, обещаной, три, год, ждать]",0.000000,0.000000,0.00,0.000000,0,0
3,"Желаю хорошего полёта и удачной посадки,я буду...","[желаю, хорошего, полёта, удачной, посадки, бу...",желаю хорошего полёта удачной посадки буду оче...,-1,11,9,"[1.0, 1.22222222222222, 0.0]","[хороший, удачный, посадка]","[желать, хороший, полёт, удачный, посадка, быт...",0.740741,1.222222,0.00,2.222222,2,0
4,"Обновил за каким-то лешим surf, теперь не рабо...","[обновил, каким, то, лешим, теперь, не, работа...",обновил каким то лешим теперь не работает прос...,-1,10,8,[0.0],[работать],"[обновить, какой, то, леший, теперь, не, работ...",0.000000,0.000000,0.00,0.000000,0,0
5,"Котёнка вчера носик разбила, плакала и расстра...","[котёнка, вчера, носик, разбила, плакала, расс...",котёнка вчера носик разбила плакала расстраива...,-1,8,6,[-1.25],[расстраиваться],"[котёнок, вчера, носик, разбить, плакать, расс...",-1.250000,-1.250000,-1.25,-1.250000,0,1
6,"@juliamayko @O_nika55 @and_Possum Зашли, а то ...","[зашли, то, он, опять, затихарился, прямо, физ...",зашли то он опять затихарился прямо физически ...,-1,17,12,[0],[],"[заслать, то, он, опять, затихариться, прямо, ...",0.000000,0.000000,0.00,0.000000,0,0


In [83]:
features_to_save = df_all_twits_text[['tone_mean', 'tone_max', 'tone_min', 'tone_sum', 
                                 'tone_pos_count', 'tone_neg_count']]

# Сохраняем в CSV
features_to_save.to_csv('tonal_features.csv', index=False, sep=';', encoding='utf-8')

# 4 Извлечь морфологические признаки с использованием библиотеки pymorphy3. 

Для каждого сообщения вычислить относительную частоту основных частей речи. Сохранить признаки в файл.

In [84]:
from collections import Counter

In [85]:
pos_tags_ru = {
    'NOUN': 'существительное',
    'ADJF': 'прилагательное (полное)',
    'ADJS': 'прилагательное (краткое)',
    'COMP': 'сравнительная форма',
    'VERB': 'глагол (личная форма)',
    'INFN': 'глагол (инфинитив)',
    'PRTF': 'причастие (полное)',
    'PRTS': 'причастие (краткое)',
    'GRND': 'деепричастие',
    'NUMR': 'числительное',
    'ADVB': 'наречие',
    'NPRO': 'местоимение',
    'PRED': 'предикатив',
    'PREP': 'предлог',
    'CONJ': 'союз',
    'PRCL': 'частица',
    'INTJ': 'междометие'
}

In [86]:
# Возьмем основные части речи
main_pos_tags = ['NOUN', 'ADJF', 'VERB', 'INFN', 'ADVB', 'NPRO', 'PREP', 'CONJ', 'PRCL', 'INTJ']

def get_pos_tags(tokens): # части речи для каждого токена в твите
    pos_list = []
    
    for token in tokens:
        try:
            parsed = morph.parse(token)[0]
            pos = parsed.tag.POS
            
            if pos is None:
                pos = 'UNKN' #если не знает часть речи
            pos_list.append(pos)
        except:
            pos_list.append('UNKN')
    
    return pos_list

In [87]:
#Относительные частоты частей речи в твите + создает словарь {ЧР: частота}
def calculate_pos_frequencies(tokens):
    if not tokens:
        return {pos: 0.0 for pos in main_pos_tags}
    
    pos_tags = get_pos_tags(tokens) #для всех токенов
    
    pos_counter = Counter(pos_tags) #сколько каждой ЧР
    
    total_words = len(tokens)
    pos_frequencies = {} #относительные частоты
    
    for pos in main_pos_tags:
        pos_frequencies[pos] = pos_counter.get(pos, 0) / total_words
    
    return pos_frequencies

In [88]:
# Применяем ко всем твитам

pos_frequencies_list = [] #список для словарей

for idx, row in df_all_twits_text.iterrows():
    tokens = row['tokens']
    freq_dict = calculate_pos_frequencies(tokens)
    pos_frequencies_list.append(freq_dict)
    
    # Прогресс (каждые 100 твитов)
    if (idx + 1) % 1000 == 0:
        print(f"Обработано {idx + 1} твитов...")

pos_df = pd.DataFrame(pos_frequencies_list)

# Обогатим основной датафрейм
for pos in main_pos_tags:
    df_all_twits_text[f'pos_{pos}'] = pos_df[pos]

Обработано 1000 твитов...
Обработано 2000 твитов...
Обработано 3000 твитов...
Обработано 4000 твитов...
Обработано 5000 твитов...
Обработано 6000 твитов...
Обработано 7000 твитов...
Обработано 8000 твитов...
Обработано 9000 твитов...
Обработано 10000 твитов...
Обработано 11000 твитов...
Обработано 12000 твитов...
Обработано 13000 твитов...
Обработано 14000 твитов...
Обработано 15000 твитов...
Обработано 16000 твитов...
Обработано 17000 твитов...
Обработано 18000 твитов...
Обработано 19000 твитов...
Обработано 20000 твитов...
Обработано 21000 твитов...
Обработано 22000 твитов...
Обработано 23000 твитов...
Обработано 24000 твитов...
Обработано 25000 твитов...
Обработано 26000 твитов...
Обработано 27000 твитов...
Обработано 28000 твитов...
Обработано 29000 твитов...
Обработано 30000 твитов...
Обработано 31000 твитов...
Обработано 32000 твитов...
Обработано 33000 твитов...
Обработано 34000 твитов...
Обработано 35000 твитов...
Обработано 36000 твитов...
Обработано 37000 твитов...
Обработано

In [90]:
print("ПРИМЕРЫ МОРФОЛОГИЧЕСКИХ ПРИЗНАКОВ:")


for i in range(min(5, len(df_all_twits_text))):
    print(f"\n--- Твит {i+1} ---")
    print(f"Текст: {df_all_twits_text.loc[i, 'text'][:100]}...")
    print(f"Токены: {df_all_twits_text.loc[i, 'tokens']}")
    
    # части речи для этого твита
    pos_tags_example = get_pos_tags(df_all_twits_text.loc[i, 'tokens'])
    print(f"Части речи: {list(zip(df_all_twits_text.loc[i, 'tokens'], pos_tags_example))}")
    
    print("Относительные частоты частей речи:")
    for pos in main_pos_tags:
        freq = df_all_twits_text.loc[i, f'pos_{pos}']
        if freq > 0:  # показываем только ненулевые
            pos_name_ru = pos_tags_ru.get(pos, pos)
            print(f"  {pos_name_ru} ({pos}): {freq:.3f}")

# Статистика по частям речи

for pos in main_pos_tags:
    col_name = f'pos_{pos}'
    pos_name_ru = pos_tags_ru.get(pos, pos)
    non_zero = (df_all_twits_text[col_name] > 0).sum()
    mean_freq = df_all_twits_text[col_name].mean()
    print(f"{pos_name_ru} ({pos}):")
    print(f"  Средняя частота: {mean_freq:.3f}")
    print(f"  Твитов с этой частью речи: {non_zero} ({non_zero/len(df_all_twits_text)*100:.1f}%)")

# СОХРАНЯЕМ только колонки с морфологическими признаками
morph_features = [f'pos_{pos}' for pos in main_pos_tags]
morph_features_to_save = df_all_twits_text[morph_features]

# Сохраняем в CSV
morph_features_to_save.to_csv('morphological_features.csv', index=False, sep=';', encoding='utf-8')


ПРИМЕРЫ МОРФОЛОГИЧЕСКИХ ПРИЗНАКОВ:

--- Твит 1 ---
Текст: на работе был полный пиддес :| и так каждое закрытие месяца, я же свихнусь так D:...
Токены: ['на', 'работе', 'был', 'полный', 'пиддес', 'так', 'каждое', 'закрытие', 'месяца', 'же', 'свихнусь', 'так']
Части речи: [('на', 'PREP'), ('работе', 'NOUN'), ('был', 'VERB'), ('полный', 'ADJF'), ('пиддес', 'NOUN'), ('так', 'CONJ'), ('каждое', 'ADJF'), ('закрытие', 'NOUN'), ('месяца', 'NOUN'), ('же', 'PRCL'), ('свихнусь', 'VERB'), ('так', 'CONJ')]
Относительные частоты частей речи:
  существительное (NOUN): 0.333
  прилагательное (полное) (ADJF): 0.167
  глагол (личная форма) (VERB): 0.167
  предлог (PREP): 0.083
  союз (CONJ): 0.167
  частица (PRCL): 0.083

--- Твит 2 ---
Текст: Коллеги сидят рубятся в Urban terror, а я из-за долбанной винды не могу :(...
Токены: ['коллеги', 'сидят', 'рубятся', 'долбанной', 'винды', 'не', 'могу']
Части речи: [('коллеги', 'NOUN'), ('сидят', 'VERB'), ('рубятся', 'VERB'), ('долбанной', 'PRTF'), ('винды', 'NO

In [91]:
# Узнаем, куда Gensim сохраняет модели
import gensim.downloader as download_api
import os

# Показываем путь к кэшу Gensim
cache_dir = download_api.base_dir
print(f"Модели Gensim сохраняются в: {cache_dir}")

# Конкретно для нашей модели
model_path = os.path.join(cache_dir, 'word2vec-ruscorpora-300', 'word2vec-ruscorpora-300.gz')
print(f"Путь к скачанной модели: {model_path}")

# Проверим, существует ли файл
if os.path.exists(model_path):
    print(f"✅ Файл существует! Размер: {os.path.getsize(model_path) / (1024**3):.2f} ГБ")
else:
    print("❌ Файл не найден по указанному пути")

Модели Gensim сохраняются в: C:\Users\Katya/gensim-data
Путь к скачанной модели: C:\Users\Katya/gensim-data\word2vec-ruscorpora-300\word2vec-ruscorpora-300.gz
✅ Файл существует! Размер: 0.19 ГБ


Извлечь признаки с использованием метода «Мешок слов», используя библиотеку sklearn (sklearn.feature_extraction.text.TfidfVectorizer). Сохранить признаки в файл (сократить размерность до 1000).

# 1) TF-IDF

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [92]:
#чистый текст - добавить в датафрейм
df_all_twits_text['text_clean'] = df_all_twits_text['tokens'].apply(lambda x: ' '.join(x))

In [93]:
# Векторизация
vectorizer = TfidfVectorizer(
    max_features=1000,        # сокращаем до 1000 признаков
    ngram_range=(1, 2),       # униграммы и биграммы
    min_df=2,                 
    max_df=0.9,               
    analyzer='word',          
    token_pattern=r'(?u)\b\w+\b',  
    lowercase=True            
)

In [94]:
#Обучение веткоризатора
X_tfidf = vectorizer.fit_transform(df_all_twits_text['text_clean'])


In [95]:
print(f"Размер полученной матрицы: {X_tfidf.shape}")
print(f"  - строк (твитов): {X_tfidf.shape[0]}")
print(f"  - колонок (признаков): {X_tfidf.shape[1]} (должно быть 1000)")

Размер полученной матрицы: (226834, 1000)
  - строк (твитов): 226834
  - колонок (признаков): 1000 (должно быть 1000)


In [96]:
feature_names = vectorizer.get_feature_names_out()

In [97]:
print(f"\nПримеры признаков (первые 20):")
for i, name in enumerate(feature_names[:50]):
    print(f"  {i+1}. {name}")
# русские слова только оставлены на уровне предобработки, свернут смех


Примеры признаков (первые 20):
  1. безумно
  2. бесит
  3. бл
  4. блин
  5. бля
  6. блять
  7. боже
  8. более
  9. болеть
  10. болею
  11. болит
  12. боль
  13. больно
  14. больше
  15. больше не
  16. большое
  17. большой
  18. боюсь
  19. брат
  20. будем
  21. будет
  22. будешь
  23. будто
  24. буду
  25. будут
  26. будь
  27. бы
  28. бы не
  29. бывает
  30. был
  31. была
  32. были
  33. было
  34. было бы
  35. быстро
  36. быть
  37. вам
  38. вами
  39. вас
  40. ваще
  41. вдруг
  42. ведь
  43. везде
  44. верю
  45. весело
  46. весь
  47. весь день
  48. вечер
  49. вечера
  50. вечером


In [98]:
# датафрейм с TF-IDF признаками
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=[f'tfidf_{i}' for i in range(X_tfidf.shape[1])] )

In [99]:
#словарь соответствия индексов и реальных слов (для информации)
word_mapping = {f'tfidf_{i}': feature_names[i] for i in range(len(feature_names))}

In [100]:
for i in range(min(3, len(tfidf_df))):
    print(f"\n--- Твит {i+1} ---")
    print(f"Текст: {df_all_twits_text.loc[i, 'text'][:100]}...")
    
    # Найдем ненулевые признаки для этого твита
    non_zero = tfidf_df.iloc[i][tfidf_df.iloc[i] > 0]
    print(f"Количество ненулевых признаков: {len(non_zero)}")
    
    if len(non_zero) > 0:
        print("Топ-5 самых важных слов в этом твите:")
        # Сортируем по убыванию и берем топ-5
        top_features = non_zero.sort_values(ascending=False).head(5)
        for idx, value in top_features.items():
            word = word_mapping[idx]
            print(f"  {word}: {value:.4f}")

# Статистика по TF-IDF матрице
print(f"Среднее количество ненулевых признаков на твит: {(X_tfidf != 0).sum(axis=1).mean():.2f}")
print(f"Максимум ненулевых признаков в твите: {(X_tfidf != 0).sum(axis=1).max()}")
print(f"Минимум ненулевых признаков в твите: {(X_tfidf != 0).sum(axis=1).min()}")

# Сохраняем TF-IDF признаки
tfidf_df.to_csv('tfidf_features.csv', index=False, sep=';', encoding='utf-8')

print(f"✅ TF-IDF признаки сохранены в файл 'tfidf_features.csv'") # проверить и сделать таблицу с признаками и вывести ???
print(f"Размер файла: {tfidf_df.shape}")
print(f"Количество признаков: {tfidf_df.shape[1]}")

# Сохраняем словарь соответствия индексов и слов (для информации)
mapping_df = pd.DataFrame(list(word_mapping.items()), columns=['column_name', 'word'])
mapping_df.to_csv('tfidf_word_mapping.csv', index=False, sep=';', encoding='utf-8')
print(f"✅ Словарь слов сохранен в файл 'tfidf_word_mapping.csv'")


# Суммируем TF-IDF значения по всем документам для каждого слова
word_importance = X_tfidf.sum(axis=0).A1  # в одномерный массив

top_indices = word_importance.argsort()[-20:][::-1]
print("Топ-20 самых важных слов в корпусе:")
for i, idx in enumerate(top_indices):
    word = feature_names[idx]
    importance = word_importance[idx]
    print(f"  {i+1}. {word}: {importance:.2f}")


--- Твит 1 ---
Текст: на работе был полный пиддес :| и так каждое закрытие месяца, я же свихнусь так D:...
Количество ненулевых признаков: 7
Топ-5 самых важных слов в этом твите:
  месяца: 0.4689
  на работе: 0.4397
  так: 0.4363
  работе: 0.4296
  был: 0.3364

--- Твит 2 ---
Текст: Коллеги сидят рубятся в Urban terror, а я из-за долбанной винды не могу :(...
Количество ненулевых признаков: 3
Топ-5 самых важных слов в этом твите:
  не могу: 0.6914
  могу: 0.6596
  не: 0.2950

--- Твит 3 ---
Текст: @elina_4post как говорят обещаного три года ждут...((...
Количество ненулевых признаков: 4
Топ-5 самых важных слов в этом твите:
  говорят: 0.5791
  три: 0.5664
  года: 0.5134
  как: 0.2832
Среднее количество ненулевых признаков на твит: 5.72
Максимум ненулевых признаков в твите: 25
Минимум ненулевых признаков в твите: 0
✅ TF-IDF признаки сохранены в файл 'tfidf_features.csv'
Размер файла: (226834, 1000)
Количество признаков: 1000
✅ Словарь слов сохранен в файл 'tfidf_word_mapping.csv'
Топ-2

In [102]:
df_all_twits_text.head(5)

,text,tokens,cleaned_text,tone,original_tokens,cleaned_tokens_count,tonal_list,found_lemmas,lemmas,tone_mean,...,pos_ADJF,pos_VERB,pos_INFN,pos_ADVB,pos_NPRO,pos_PREP,pos_CONJ,pos_PRCL,pos_INTJ,text_clean
0,на работе был полный пиддес :| и так каждое за...,"[на, работе, был, полный, пиддес, так, каждое,...",на работе был полный пиддес так каждое закрыти...,-1,16,12,"[0.166666666666667, 0.222222222222222, 0.0]","[работа, полный, закрытие]","[на, работа, быть, полный, пиддес, так, каждый...",0.129630,...,0.166667,0.166667,0.000000,0.000000,0.0,0.083333,0.166667,0.083333,0.0,на работе был полный пиддес так каждое закрыти...
1,"Коллеги сидят рубятся в Urban terror, а я из-з...","[коллеги, сидят, рубятся, долбанной, винды, не...",коллеги сидят рубятся долбанной винды не могу,-1,14,7,[-0.2],[сидеть],"[коллега, сидеть, рубиться, долбать, винд, не,...",-0.200000,...,0.000000,0.428571,0.000000,0.000000,0.0,0.000000,0.000000,0.142857,0.0,коллеги сидят рубятся долбанной винды не могу
2,@elina_4post как говорят обещаного три года жд...,"[как, говорят, обещаного, три, года, ждут]",как говорят обещаного три года ждут,-1,7,6,[0.0],[ждать],"[как, говорить, обещаной, три, год, ждать]",0.000000,...,0.166667,0.333333,0.000000,0.000000,0.0,0.000000,0.166667,0.000000,0.0,как говорят обещаного три года ждут
3,"Желаю хорошего полёта и удачной посадки,я буду...","[желаю, хорошего, полёта, удачной, посадки, бу...",желаю хорошего полёта удачной посадки буду оче...,-1,11,9,"[1.0, 1.22222222222222, 0.0]","[хороший, удачный, посадка]","[желать, хороший, полёт, удачный, посадка, быт...",0.740741,...,0.222222,0.222222,0.111111,0.222222,0.0,0.000000,0.000000,0.000000,0.0,желаю хорошего полёта удачной посадки буду оче...
4,"Обновил за каким-то лешим surf, теперь не рабо...","[обновил, каким, то, лешим, теперь, не, работа...",обновил каким то лешим теперь не работает прос...,-1,10,8,[0.0],[работать],"[обновить, какой, то, леший, теперь, не, работ...",0.000000,...,0.125000,0.250000,0.000000,0.125000,0.0,0.000000,0.125000,0.125000,0.0,обновил каким то лешим теперь не работает прос...


In [ ]:
#для второй части

In [103]:
import pickle
import pymorphy3
from tqdm import tqdm

# Создаем анализатор
morph = pymorphy3.MorphAnalyzer()

# Словарь для преобразования тегов pymorphy3 в теги для модели
pos_mapping = {
    'NOUN': 'NOUN',      # существительное
    'ADJF': 'ADJ',       # прилагательное
    'ADJS': 'ADJ',       # краткое прилагательное
    'VERB': 'VERB',      # глагол
    'INFN': 'VERB',      # инфинитив
    'PRTF': 'ADJ',       # причастие
    'PRTS': 'ADJ',       # краткое причастие
    'GRND': 'VERB',      # деепричастие
    'NUMR': 'NUM',       # числительное
    'ADVB': 'ADV',       # наречие
    'NPRO': 'PRON',      # местоимение
    'PREP': 'ADP',       # предлог
    'CONJ': 'CCONJ',     # союз
    'PRCL': 'PART',      # частица
    'INTJ': 'INTJ',      # междометие
}

def get_pos_tag(token):
    """
    Определяет часть речи для токена и возвращает в формате слово_ЧАСТЬ_РЕЧИ
    """
    try:
        parsed = morph.parse(token)[0]
        pos = parsed.tag.POS
        if pos and pos in pos_mapping:
            model_pos = pos_mapping[pos]
            return f"{token}_{model_pos}"
        else:
            return f"{token}_X"
    except:
        return f"{token}_X"

def create_sentiment_pos_data(df):
    """
    Создает структуру данных для sentiment_POS из датафрейма
    """
    print("Создаем частеречную разметку для всех твитов...")
    
    ttext = []      # исходные тексты
    ttype = []       # тональность
    text_pos = []    # текст с разметкой (строка)
    X = []           # список токенов с разметкой
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        # Исходный текст
        ttext.append(row['text'])
        
        # Тональность (преобразуем -1/1 в 0/1)
        tone = 0 if row['tone'] == -1 else 1
        ttype.append(tone)
        
        # Получаем размеченные токены из нашей колонки 'tokens'
        tokens = row['tokens']
        pos_tokens = [get_pos_tag(token) for token in tokens]
        
        # Сохраняем список размеченных токенов
        X.append(pos_tokens)
        
        # Создаем строку с разметкой (через пробел)
        text_pos.append(' '.join(pos_tokens))
    
    # Создаем датафрейм
    result_df = pd.DataFrame({
        'ttext': ttext,
        'ttype': ttype,
        'text_pos': text_pos,
        'X': X
    })
    
    return result_df

# Создаем данные
sentiment_pos_df = create_sentiment_pos_data(df_all_twits_text)

# Покажем результат
print("\n" + "="*80)
print("РЕЗУЛЬТАТ:")
print("="*80)
print(f"Создано {len(sentiment_pos_df)} записей")
print(f"Колонки: {sentiment_pos_df.columns.tolist()}")

print("\nПример первых 3 строк:")
for i in range(3):
    print(f"\n--- Твит {i+1} ---")
    print(f"ttext: {sentiment_pos_df.loc[i, 'ttext'][:100]}...")
    print(f"ttype: {sentiment_pos_df.loc[i, 'ttype']}")
    print(f"text_pos: {sentiment_pos_df.loc[i, 'text_pos'][:100]}...")
    print(f"X (первые 5 токенов): {sentiment_pos_df.loc[i, 'X'][:5]}")

  0%|                                                                                       | 0/226834 [00:00<?, ?it/s]

Создаем частеречную разметку для всех твитов...


100%|█████████████████████████████████████████████████████████████████████████| 226834/226834 [08:20<00:00, 453.62it/s]



РЕЗУЛЬТАТ:
Создано 226834 записей
Колонки: ['ttext', 'ttype', 'text_pos', 'X']

Пример первых 3 строк:

--- Твит 1 ---
ttext: на работе был полный пиддес :| и так каждое закрытие месяца, я же свихнусь так D:...
ttype: 0
text_pos: на_ADP работе_NOUN был_VERB полный_ADJ пиддес_NOUN так_CCONJ каждое_ADJ закрытие_NOUN месяца_NOUN же...
X (первые 5 токенов): ['на_ADP', 'работе_NOUN', 'был_VERB', 'полный_ADJ', 'пиддес_NOUN']

--- Твит 2 ---
ttext: Коллеги сидят рубятся в Urban terror, а я из-за долбанной винды не могу :(...
ttype: 0
text_pos: коллеги_NOUN сидят_VERB рубятся_VERB долбанной_ADJ винды_NOUN не_PART могу_VERB...
X (первые 5 токенов): ['коллеги_NOUN', 'сидят_VERB', 'рубятся_VERB', 'долбанной_ADJ', 'винды_NOUN']

--- Твит 3 ---
ttext: @elina_4post как говорят обещаного три года ждут...((...
ttype: 0
text_pos: как_CCONJ говорят_VERB обещаного_ADJ три_NUM года_NOUN ждут_VERB...
X (первые 5 токенов): ['как_CCONJ', 'говорят_VERB', 'обещаного_ADJ', 'три_NUM', 'года_NOUN']


In [104]:
# Сохраняем в файл sentiment_POS
with open('sentiment_POS', 'wb') as f:
    pickle.dump(sentiment_pos_df, f)

print("✅ Файл 'sentiment_POS' успешно создан и сохранен!")

# Проверим размер файла
import os
file_size = os.path.getsize('sentiment_POS') / (1024 * 1024)  # в МБ
print(f"Размер файла: {file_size:.2f} МБ")

✅ Файл 'sentiment_POS' успешно создан и сохранен!
Размер файла: 102.24 МБ


In [106]:
# Загружаем файл обратно и проверяем
with open('sentiment_POS', 'rb') as f:
    loaded_data = pickle.load(f)

print("Проверка загруженных данных:")
print(f"Тип: {type(loaded_data)}")
print(f"Размер: {loaded_data.shape}")
print(f"Колонки: {loaded_data.columns.tolist()}")

print("\nПервые 3 строки из загруженного файла:")
print(loaded_data[['ttext', 'ttype', 'text_pos']].head(3))

Проверка загруженных данных:
Тип: <class 'pandas.core.frame.DataFrame'>
Размер: (226834, 4)
Колонки: ['ttext', 'ttype', 'text_pos', 'X']

Первые 3 строки из загруженного файла:
                                               ttext  ttype  \
0  на работе был полный пиддес :| и так каждое за...      0   
1  Коллеги сидят рубятся в Urban terror, а я из-з...      0   
2  @elina_4post как говорят обещаного три года жд...      0   

                                            text_pos  
0  на_ADP работе_NOUN был_VERB полный_ADJ пиддес_...  
1  коллеги_NOUN сидят_VERB рубятся_VERB долбанной...  
2  как_CCONJ говорят_VERB обещаного_ADJ три_NUM г...  
